# Bias-Variance Decomposition

Companion notebook for the wiki page: [Bias-Variance Decomposition](https://ml-viz-ruby.vercel.app/wiki/bias-variance-decomposition)

We empirically verify the decomposition  
**E[(y − f̂)²] = Bias² + Variance + σ²**  
by running KNN on many resamples of a synthetic dataset and measuring bias and variance as k varies.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'font.size': 11,
})

## 1 · The synthetic dataset

True function: **f(x) = sin(2πx)** on [0,1] with additive Gaussian noise σ=0.3.  
We measure bias² and variance at a single test point x₀ = 0.5 over M=300 training resamples.

In [ ]:
rng = np.random.default_rng(42)

def f_true(x):
    return np.sin(2 * np.pi * x)

n_train = 60      # points per training set
M = 300           # number of resampled training sets
sigma_noise = 0.3
x_test = np.array([[0.5]])   # single evaluation point
f_star = f_true(0.5)         # true f(x₀)

# Visualise one draw
X_demo = rng.uniform(0, 1, (n_train, 1))
y_demo = f_true(X_demo).ravel() + rng.normal(0, sigma_noise, n_train)
x_grid = np.linspace(0, 1, 200)

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X_demo, y_demo, s=15, alpha=0.6, color='#94a3b8', label='noisy data')
ax.plot(x_grid, f_true(x_grid), color='#6366f1', lw=2, label='f(x) = sin(2πx)')
ax.axvline(0.5, color='#f59e0b', ls='--', alpha=0.7, label='x₀ = 0.5')
ax.set(xlabel='x', ylabel='y', title='One training draw')
ax.legend()
plt.tight_layout()
plt.show()

## 2 · Empirical bias² and variance across k

For each k, fit KNN on M resamples, record predictions at x₀:

- **Variance** = average squared deviation from the mean prediction
- **Bias²** = squared deviation of mean prediction from f(x₀)
- **Total** = Bias² + Variance (ignoring σ² which is constant)

In [ ]:
k_values = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50, 60]
results = []

for k in k_values:
    preds = []
    for _ in range(M):
        X = rng.uniform(0, 1, (n_train, 1))
        y = f_true(X).ravel() + rng.normal(0, sigma_noise, n_train)
        model = KNeighborsRegressor(n_neighbors=k)
        model.fit(X, y)
        preds.append(model.predict(x_test)[0])
    preds = np.array(preds)
    mean_pred = preds.mean()
    bias2 = (mean_pred - f_star) ** 2
    variance = preds.var()
    results.append({'k': k, 'bias2': bias2, 'var': variance, 'total': bias2 + variance})

# Print table
print(f"{'k':>4}  {'Bias²':>8}  {'Variance':>10}  {'Total':>8}")
print("-" * 38)
for r in results:
    print(f"{r['k']:>4}  {r['bias2']:>8.4f}  {r['var']:>10.4f}  {r['total']:>8.4f}")

In [ ]:
ks      = [r['k']     for r in results]
bias2s  = [r['bias2'] for r in results]
vars_   = [r['var']   for r in results]
totals  = [r['total'] for r in results]

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(ks, bias2s,  'o-', color='#f59e0b', lw=2, ms=6, label='Bias²')
ax.plot(ks, vars_,   's-', color='#6366f1', lw=2, ms=6, label='Variance')
ax.plot(ks, totals,  '^-', color='#34d399', lw=2, ms=6, label='Total (Bias² + Var)')

best_k = ks[np.argmin(totals)]
ax.axvline(best_k, color='#94a3b8', ls='--', alpha=0.5, label=f'Optimal k={best_k}')

ax.set(xlabel='k (number of neighbors)', ylabel='Error component',
       title='Bias–Variance Tradeoff in KNN')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Optimal k = {best_k} (minimum total error = {min(totals):.4f})")
print(f"σ² (irreducible) ≈ {sigma_noise**2:.4f}")

## 3 · Visualise the spread of fits at k=1 vs k=15

Low k → high variance (fits jump around).  
High k → high bias (fits miss the curve).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x_grid2d = np.linspace(0, 1, 200).reshape(-1, 1)

for ax, k, title in zip(axes, [1, 15], ['k=1 (high variance)', 'k=15 (higher bias)']):
    for _ in range(30):
        X = rng.uniform(0, 1, (n_train, 1))
        y = f_true(X).ravel() + rng.normal(0, sigma_noise, n_train)
        pred = KNeighborsRegressor(n_neighbors=k).fit(X, y).predict(x_grid2d)
        ax.plot(x_grid2d, pred, color='#6366f1', alpha=0.15, lw=1)
    ax.plot(x_grid2d, f_true(x_grid2d.ravel()), color='#f59e0b', lw=2.5, label='f(x) = sin(2πx)')
    ax.set(title=title, xlabel='x', ylabel='f̂(x)')
    ax.legend()

plt.suptitle('30 KNN fits on different training sets — spread = variance', y=1.02)
plt.tight_layout()
plt.show()

---

## ✏️ Your turn

### Exercise 1 — change the noise level

Set `sigma_noise = 0.1` (low noise) and `sigma_noise = 0.8` (high noise). Rerun the experiment.  
**Prediction:** the irreducible noise σ² changes, but the *optimal k* (the U-curve minimum) should stay roughly the same. Verify this.

In [ ]:
# TODO(you): copy the loop above, change sigma_noise, compare the U-curves

sigma_low  = 0.1
sigma_high = 0.8

# ... your code here ...

### Exercise 2 — decomposition check

For the optimal k found above, verify that `Bias² + Variance + σ²` matches the empirical MSE on held-out points (not just x₀).

In [ ]:
# TODO(you): generate a test set of 200 points, average the MSE over all of them,
# and compare to Bias² + Variance + sigma_noise**2

X_test_full = np.linspace(0, 1, 200).reshape(-1, 1)
# ... your code here ...

In [ ]:
# Assert: the two numbers should be close (within Monte Carlo noise)
# assert abs(empirical_mse - (bias2 + variance + sigma_noise**2)) < 0.05, "decomposition mismatch"
print("Uncomment the assert after filling in the code above.")

<details>
<summary>Solution sketch</summary>

```python
k_opt = ks[np.argmin(totals)]
X_test_full = np.linspace(0, 1, 200).reshape(-1, 1)
f_test = f_true(X_test_full.ravel())

mse_list = []
for _ in range(M):
    X = rng.uniform(0, 1, (n_train, 1))
    y = f_true(X).ravel() + rng.normal(0, sigma_noise, n_train)
    preds = KNeighborsRegressor(n_neighbors=k_opt).fit(X, y).predict(X_test_full)
    # Add noise to targets
    y_test_noisy = f_test + rng.normal(0, sigma_noise, len(f_test))
    mse_list.append(np.mean((y_test_noisy - preds)**2))

empirical_mse = np.mean(mse_list)
print(f"Empirical MSE: {empirical_mse:.4f}")
print(f"Bias²+Var+σ² : {results[ks.index(k_opt)]['total'] + sigma_noise**2:.4f}")
```
</details>